# GraphAgents: Knowledge Graph-Guided Agentic AI for Cross-Domain Materials Design

#### Authors: Isabella Stewart, Tarjei Hage, Yu-Chuan (Michael) Hsu, and Markus J. Buehler, MIT, 2025 
#### Corresponding author: Markus J. Buehler, mbuehler@MIT.EDU
#### LAMM, Massachusetts Institute of Technology

In [ ]:
import sys, os

try:
    thread_i = int(sys.argv[1])
    total_threads = int(sys.argv[2])

except: #If you don’t provide any arguments, it defaults to thread_i=0 and total_threads=1 — meaning it will process everything by itself.
    thread_i = 0
    total_threads = 1
    merge_every = 500

In [ ]:
config_list = [
    {
        "model":"meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
        # "model":"meta-llama/Llama-3.3-70B-Instruct-Turbo",
        # "model":"meta-llama/Llama-3.3-70B-Instruct-Turbo-Free",
        
        "api_key":os.getenv("Llama4_together"),
        "max_tokens": 20000
    },
]

In [ ]:
from together import Together
client = Together(api_key=config_list[0]["api_key"])


In [ ]:
import os
from GraphReasoning import *



In [ ]:
verbatim=False

In [ ]:
doc_data_dir = '/orcd/pool/005/mkychsu/SG_materialproperties'
data_dir='./GRAPHDATA'    
data_dir_output='./GRAPHDATA_OUTPUT'

max_tokens = config_list[0]['max_tokens']

embedding_file='SG_LLAMA4_70b.pkl'


In [ ]:
if total_threads == 1: ##merging mode
  
    from transformers import AutoModelForCausalLM, AutoTokenizer
    # from tqdm.notebook import tqdm
    # from IPython.display import display, Markdown
    
    
    # tokenizer_model=f'/home/mkychsu/pool/llm/SEMIKONG-8b-GPTQ'
    tokenizer_model=f'/home/mkychsu/pool/llm/nomic-embed-text-v1.5'
    
    # embedding_tokenizer = AutoTokenizer.from_pretrained(tokenizer_model,use_fast=False, device_map="cuda:0")
    # embedding_model = AutoModelForCausalLM.from_pretrained(tokenizer_model,output_hidden_states=True).to('cuda:0')
    
    from sentence_transformers import SentenceTransformer
    embedding_tokenizer =''
    embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
    
    
    from GraphReasoning import load_embeddings, save_embeddings, generate_node_embeddings
    
    # generate_new_embeddings=True
    
    # from PIL import Image
    # from transformers import AutoModelForCausalLM 
    # from transformers import AutoProcessor 
    
    # model = AutoModelForCausalLM.from_pretrained(model_id, device_map="cuda:1", trust_remote_code=True, torch_dtype="auto")
    # processor = AutoProcessor.from_pretrained(model_id, device_map="cuda:1", trust_remote_code=True) 


    import torch
    generate_new_embeddings=True
    
    if os.path.exists(f'{data_dir}/{embedding_file}'):
        print('Found existing embedding file')
        generate_new_embeddings=False
    generate_new_embeddings=True
    
    with torch.no_grad():
        if generate_new_embeddings:
            G=nx.DiGraph()
            node_embeddings = generate_node_embeddings(G, embedding_tokenizer, embedding_model, )
            save_embeddings(node_embeddings, f'{data_dir}/{embedding_file}')
        else:
            node_embeddings = load_embeddings(f'{data_dir}/{embedding_file}')
            
            


### Load dataset

In [ ]:

### Load dataset of papers

# In[3]:

import pandas as pd
import glob
try:
    df = pd.read_csv(f'{doc_data_dir}/abstract.csv', index_col=0)
    
except:
    
    doc_list=sorted(glob.glob(f'{doc_data_dir}/*.xls'))
    df_list = []
    for i, doc in enumerate(doc_list):
        print(i, doc)
        df_list.append(pd.read_excel(doc))
        
    df = pd.concat(df_list, axis=0)
    df = df.drop_duplicates()
    df = df.reset_index(drop=True)
    df.to_csv(f'{doc_data_dir}/abstract.csv', drop_index=True)


In [ ]:
df.shape

### Set up LLM client:

In [ ]:

import instructor
from typing import List
from PIL import Image
import base64

from pydantic import BaseModel

class Node(BaseModel):
    id: str
    type: str
        
class Edge(BaseModel):
    source: str
    target: str
    relation: str
        
class KnowledgeGraph(BaseModel):
    nodes: List[Node]
    edges: List[Edge]

response_model = KnowledgeGraph
system_prompt = '''
You are a scientific assistant extracting knowledge graphs from text.
Return a JSON with two fields: <nodes> and <edges>.\n
Each node must have <id> and <type>.\n
Each edge must have <source>, <target>, and <relation>.
'''


def generate(system_prompt=system_prompt, 
             prompt="",temperature=0.333,
             max_tokens=config_list[0]['max_tokens'], response_model=KnowledgeGraph, 
            ):     

    if system_prompt==None:
        messages=[
            {"role": "user", "content": f"{prompt}"},
        ]

    else:
        messages=[
            {"role": "system",  "content": f"{system_prompt}"},
            {"role": "user", "content": f"{prompt}"},
        ]

    
    create = instructor.patch(
        create=client.chat.completions.create,
        mode=instructor.Mode.JSON_SCHEMA,
    )

    return create(messages=messages,   
                    model=config_list[0]["model"],
                    max_tokens=max_tokens,
                    temperature=0.333,
                    response_model=response_model,
                   )

def image_to_base64_data_uri(file_path):
    with open(file_path, "rb") as image_file:
        base64_data = base64.b64encode(image_file.read()).decode("utf-8")
        return f"data:image/png;base64,{base64_data}"

def generate_figure(image, system_prompt=system_prompt, 
                prompt="", temperature=0,
                ):
    try:
        pwd = os.getcwd()
        image = image.split(pwd)[-1]
        image=Path('.').glob(f'**/{image}', case_sensitive=False)
        image = list(image)[0]
    except:
        return '' 
    image_uri = image_to_base64_data_uri(image)
    
    messages = [
        {"role": "system", "content": "You are an assistant who perfectly describes images."},
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": image_uri}},
                {"type": "text", "text": "Describe this image in detail please."},
            ],
        },
    ]
        
    return create(messages=messages,   
                    model=config_list[0]["model"],
                    max_tokens=max_tokens,
                    temperature=0.333,
                    response_model=response_model,
                   ).choices[0].message.content

In [ ]:
print(f'running on {thread_i}-th thread in totally {total_threads} threads')

In [ ]:
import glob
if total_threads == 1: # merging mode
    # try:
    merged_graph_list = sorted(glob.glob(f'{data_dir_output}/*_integrated.graphml'), reverse=True, key = lambda x: int(x.split('_')[1].split('/')[-1]))
    current_merged_i = int(merged_graph_list[0].split('_')[1].split('/')[-1])
    last_graph = sorted(glob.glob(f'{data_dir_output}/{current_merged_i}_*.graphml') )
    
    # trim the working graphs from the last session if it is be broken.
    try:
        nx.read_graphml(merged_graph_list[0])
        nx.read_graphml(last_graph)
    except:
        if os.path.exists(merged_graph_list[0]):
            os.remove(merged_graph_list[0])
        last_graph_files = sorted(glob.glob(f'{data_dir}/{current_merged_i}_*'), reverse=True)
  
        for file in last_graph_files:
            if os.path.exists(file):
                os.remove(file)
        current_merged_i -= 1

    print(f'Start merging from No. {current_merged_i}')
    ckpt = (current_merged_i//100-5)*100
    print(f'Try deleting subgraphs before {ckpt}')
    for subgraph_file in merged_graph_list:
        i = int(subgraph_file.split('_')[1].split('/')[-1])
        try:
            print(f'Delete {i}: {subgraph_file}')
            if i < ckpt:
                os.remove(subgraph_file)
        except:
            pass
else:
    current_merged_i = 0


In [ ]:

import networkx as nx
from GraphReasoning import make_graph_from_text
from datetime import datetime
import time
import torch
import shutil

G=nx.DiGraph()
with torch.no_grad():
    for i, row in df.iterrows():
        if i % total_threads != thread_i:
            continue
        if i < current_merged_i:
            continue
        
        title = row['Article Title'].replace('/','|')
        doi = row['DOI']
        txt = row['Abstract']
        
        graph_root = f'{i}_{title[:min(100, len(title))]}'
        current_graph = f'{data_dir}/{graph_root}.graphml'
        # image_list = glob.glob(''.join(doc.split('/')[:-1])+'/*png') # if running for image node
    
        while not os.path.exists(current_graph):
            print(f"Generating KG for {i}: {title}")
            try:
                if type(txt) is not str:
                    print(txt)
                    break # format of abstract is wrong 
                now = datetime.now()
                _, current_graph, _, _, _ = make_graph_from_text(txt,generate,
                                      generate_figure, image_list='',
                                      graph_root=graph_root,do_distill=False,
                                      chunk_size=200000,chunk_overlap=0,
                                      repeat_refine=0,verbatim=False,
                                      data_dir=data_dir,            
                                      save_PDF=False,
                                     )
                print("Time: ", datetime.now()-now)
            # successfully generate KGs without error
            except:
                print('Reach rate limit')
                time.sleep(60)
                
        if type(txt) is not str:
            print(txt)
            continue 
        
        if total_threads == 1: # merging mode
            if i % merge_every == 0:
                do_simplify_graph = True
                size_threshold = 10
            else:
                do_simplify_graph = False
                size_threshold = 0

     
            _graph_GraphML= f'{data_dir_output}/{graph_root}_integrated.graphml' 
            if os.path.exists(_graph_GraphML): 
                G = nx.read_graphml(_graph_GraphML)
                print(f'Main KG loaded: {_graph_GraphML}')
                print(f'{G}')
                continue
            
            
            now = datetime.now()
            print(f'Merging graph No. {i}: {title} to the main one')
            
            current_graph = nx.read_graphml(current_graph)
            nx.set_edge_attributes(current_graph, doi, "DOI")
 
            _, G, _, node_embeddings, _ = add_new_subgraph_from_text(txt='',
                               node_embeddings=node_embeddings,
                               tokenizer=embedding_tokenizer,
                               model=embedding_model,
                               original_graph=G, data_dir_output=data_dir_output, graph_root=graph_root,
                               do_simplify_graph=do_simplify_graph, size_threshold=size_threshold,
                               do_update_node_embeddings=do_simplify_graph,
                               repeat_refine=0,similarity_threshold=0.9,
                               do_Louvain_on_new_graph=do_simplify_graph,
                               #whether or not to simplify, uses similiraty_threshold defined above
                               return_only_giant_component=False,
                               save_common_graph=False,G_to_add=current_graph,graph_GraphML_to_add=None,
                               verbatim=True,)
            save_embeddings(node_embeddings, f'{data_dir}/{embedding_file}')
            print("Time: ", datetime.now()-now)

